In [ ]:
import os

os.environ["PYKEOPS_VERBOSE"] = "0"
os.environ["KEOPS_VERBOSE"] = "0"

In [ ]:
import torch

# Force CUDA re-initialisation
if torch.cuda.is_available():
    torch.cuda.init()

print(torch.cuda.device_count())  # should now be > 0

In [ ]:
import sys
import os
import pickle
import logging
import warnings
import re
from typing import Dict, List
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec

import torch

import massbalancemachine as mbm

sys.path.append(os.path.join(os.getcwd(), '../../'))
from regions.TF_Europe.scripts.config_TF_Europe import *
from regions.TF_Europe.scripts.config_models import *
from regions.TF_Europe.scripts.dataset import *
from regions.TF_Europe.scripts.plotting import *
from regions.TF_Europe.scripts.models import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.cuda.is_available())
print(torch.cuda.device_count())

if torch.cuda.device_count() == 0:
    print("No GPU detected. Using CPU.")
    device = torch.device("cpu")
    
BASE_DIR_XREG = Path(cfg.dataPath) / path_cache / f"CROSS_REGION"
# make sure BASE_DIR exists
os.makedirs(BASE_DIR_XREG, exist_ok=True)
print(f"Base directory for data: {BASE_DIR_XREG}")

BASE_DIR_FD = Path(cfg.dataPath) / path_cache / f"FD_MODEL"
CACHE_DIR = BASE_DIR_FD / "datasets/"

# Cross-regional modelling:

### Read stakes datasets:

In [ ]:
"""
Examples of loading data:
# Load Switzerland only
df = load_stakes(cfg, "CH")

# Load all Central Europe (SJM+CH+IT+AT when you add them)
df_ceu = load_stakes_for_rgi_region(cfg, "11")

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}"""

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}

dfs, glacier_dict_ca = add_o2region_to_dfs(dfs, '13', cfg)
dfs, glacier_dict_alaska = add_o2region_to_dfs(dfs,
                                               '01',
                                               cfg,
                                               deduplicate_glaciers=True)

### Monthly datasets:

In [ ]:
SOURCE_REGIONS = ['CH', 'NOR', 'ISL', "FR"]
TARGET_REGIONS = ['IT_AT', 'SJM', 'CENTRALASIA', 'ALA']

run_flag_by_code = {
    'ALA': False,
    'CENTRALASIA': False,
    'CH': False,
    'ISL': False,
    'NOR': False,
    'SJM': False,
    'FR': False,
    'IT_AT': False
}
# Compute monthly data once per code
monthly_cache = build_monthly_cache(
    cfg=cfg,
    dfs=dfs,
    paths_multi=paths_multi,
    vois_climate=VOIS_CLIMATE,
    vois_topographical=STATIC_COLS,
    region_codes=SOURCE_REGIONS + TARGET_REGIONS,
    run_flag_by_code=run_flag_by_code,
)

# Map O2Region into monthly_cache for CENTRALASIA
monthly_cache['CENTRALASIA']['data_monthly']['O2Region'] = (
    monthly_cache['CENTRALASIA']['data_monthly']['GLACIER'].map({
        k:
        v['O2Region']
        for k, v in glacier_dict_ca.items()
    }))
monthly_cache['CENTRALASIA']['data_monthly_aug']['O2Region'] = (
    monthly_cache['CENTRALASIA']['data_monthly_aug']['GLACIER'].map({
        k:
        v['O2Region']
        for k, v in glacier_dict_ca.items()
    }))

# Map O2Region into monthly_cache for ALA
monthly_cache['ALA']['data_monthly']['O2Region'] = (
    monthly_cache['ALA']['data_monthly']['GLACIER'].map({
        k: v['O2Region']
        for k, v in glacier_dict_alaska.items()
    }))
monthly_cache['ALA']['data_monthly_aug']['O2Region'] = (
    monthly_cache['ALA']['data_monthly_aug']['GLACIER'].map({
        k: v['O2Region']
        for k, v in glacier_dict_alaska.items()
    }))

# IT/AT split
monthly_cache = split_monthly_cache_by_glaciers(
    monthly_cache,
    'IT_AT',
    subregions={
        'IT': IT_GLACIERS,
        'AT': AT_GLACIERS
    },
    drop_parent=True,
)

# Central Asia subregions (O2Region already mapped onto dfs['13'])
monthly_cache = split_monthly_cache_by_glaciers(
    monthly_cache,
    'CENTRALASIA',
    subregions={
        'CA_12': {
            'o2region_col': 'O2Region',
            'values': ['1', '2']
        },
        'CA_3': {
            'o2region_col': 'O2Region',
            'values': ['3']
        },
        'CA_4': {
            'o2region_col': 'O2Region',
            'values': ['4']
        },
    },
)

# Alaska subregions (O2Region already mapped onto dfs['01'])
monthly_cache = split_monthly_cache_by_glaciers(
    monthly_cache,
    'ALA',
    subregions={
        'ALA_2': {
            'o2region_col': 'O2Region',
            'values': ['2']
        },
        'ALA_4': {
            'o2region_col': 'O2Region',
            'values': ['4']
        },
        'ALA_6': {
            'o2region_col': 'O2Region',
            'values': ['6']
        },
    },
)

# With subregions
TARGET_REGIONS_SUB = [
    'IT',
    'AT',
    'SJM',
    'ALA',
    'CA_3',
    'CA_4',
]

XREG_PAIRS = [(src,
               [r for r in SOURCE_REGIONS + TARGET_REGIONS_SUB if r != src])
              for src in SOURCE_REGIONS]

# Assemble train/test pairs from cache — no ERA5 reprocessing
res_xreg_by_source, split_info_by_source = prepare_xreg_pairs_from_cache(
    cfg=cfg,
    monthly_cache=monthly_cache,
    xreg_pairs=XREG_PAIRS,
)

## ML models
### Model datasets:

In [ ]:
months_head_pad = res_xreg_by_source["CH"]['months_head_pad']
months_tail_pad = res_xreg_by_source["CH"]['months_tail_pad']

# Build glacier→region mapping from all source regions (no ranking needed)
df_all_glaciers = pd.concat(
    [
        res_xreg_by_source[region]["df_train"][[
            "GLACIER"
        ]].drop_duplicates().assign(region=region) for region in SOURCE_REGIONS
    ],
    ignore_index=True,
).rename(columns={"GLACIER": "glacier"})

# Add measurement counts (needed by build_assets_from_glacier_list internally? No — only used by build_glacier_subsets)
# df_ranked just needs columns: "glacier", "region"  ← that's all build_assets uses
glaciers_all = df_all_glaciers["glacier"].tolist()
print(f"Total unique glaciers across all source regions: {len(glaciers_all)}")

assets_full = build_assets_from_glacier_list(
    glaciers=glaciers_all,
    df_ranked=df_all_glaciers,  # acts as the gl→region lookup
    res_xreg_by_source=res_xreg_by_source,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    cfg=cfg,
    cache_path=str(CACHE_DIR / "assets_foundation_100pct.joblib"),
    force_recompute=False,
    months_head_pad=months_head_pad,
    months_tail_pad=months_tail_pad,
    src_regions=SOURCE_REGIONS,
)

### Train model:

#### Single region:

In [ ]:
XREG_MODELS_BASE = Path(cfg.dataPath) / path_cache / "CROSS_REGION/models"
TRAIN_FLAG = False

single_source_models = {}
single_source_assets = {}
single_source_scalers = {}

for src in SOURCE_REGIONS:
    ckpt = XREG_MODELS_BASE / src / f"transformer_xreg_{src}_{src}_{MODEL_DATE}.pt"
    os.makedirs(ckpt.parent, exist_ok=True)

    df_src = res_xreg_by_source[src]["df_train"][["GLACIER"]] \
               .drop_duplicates().assign(region=src) \
               .rename(columns={"GLACIER": "glacier"})
    assets_src = build_assets_from_glacier_list(
        glaciers=df_src["glacier"].tolist(),
        df_ranked=df_src,
        res_xreg_by_source=res_xreg_by_source,
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
        cfg=cfg,
        cache_path=str(XREG_MODELS_BASE / src / f"assets_{src}_100pct.joblib"),
        force_recompute=False,
        months_head_pad=months_head_pad,
        months_tail_pad=months_tail_pad,
        src_regions=[src],
    )
    single_source_assets[src] = assets_src

    # build per-source scaler
    scaler_src = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        assets_src["ds_train"])
    scaler_src.make_loaders(
        train_idx=assets_src["train_idx"],
        val_idx=assets_src["val_idx"],
        fit_and_transform=True,
        seed=cfg.seed,
        verbose=False,
    )
    single_source_scalers[src] = scaler_src

    m = mbm.models.Transformer_MB.build_model_from_params(cfg,
                                                          PARAMS_TRANSFORMER,
                                                          device,
                                                          verbose=False)
    if ckpt.exists():
        state = torch.load(ckpt, map_location=device)
        m.load_state_dict(state)
        print(f"Loaded {src} model from {ckpt}")
    else:
        print(f"No checkpoint found for {src}, training from scratch...")
        m, _, _ = train_or_load_one_source_model(
            cfg=cfg,
            key=src,
            lstm_assets=assets_src,
            best_params=PARAMS_TRANSFORMER,
            device=device,
            models_dir=XREG_MODELS_BASE / src,
            prefix="transformer_xreg",
            train_flag=TRAIN_FLAG,
            force_retrain=False,
            epochs=N_EPOCHS,
            date=MODEL_DATE,
            model_type="transformer",
            verbose=False,
        )
        print(f"Trained {src} model, saved to {ckpt}")

    single_source_models[src] = m

#### Pooled transformer:

In [ ]:
TRAIN_FLAG = False
models_dir = BASE_DIR_FD / "models" / "foundation"
os.makedirs(models_dir, exist_ok=True)

model_pooled, model_path, info = train_or_load_one_source_model(
    cfg=cfg,
    key="foundation_100pct",
    lstm_assets=assets_full,
    best_params=PARAMS_TRANSFORMER,
    device=device,
    models_dir=models_dir,
    prefix="transformer_foundation",
    train_flag=TRAIN_FLAG,
    force_retrain=False,
    epochs=N_EPOCHS,
    date=MODEL_DATE,
    model_type="transformer",
    verbose=True,
)

print(f"Pooled model saved to: {model_path}")

# Build scaler once outside the loop
ds_pooled_scaler = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
    assets_full["ds_train"])
ds_pooled_scaler.make_loaders(
    train_idx=assets_full["train_idx"],
    val_idx=assets_full["val_idx"],
    fit_and_transform=True,
    seed=cfg.seed,
    verbose=False,
)

#### Fine tuned transformer:

In [ ]:
import json

# ── Always compute scalers/blurs — needed for any recomputation ───────────────
res_source_only = {
    region: {
        "df_train": res_xreg_by_source[region]["df_train"],
    }
    for region in SOURCE_REGIONS
}

scaler_m, scaler_s, scaler_joint, _ = build_global_scalers_multi_source_simple(
    res_source_only,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
)
blur_m, blur_s, blur_joint, _ = estimate_global_bandwidths_simple(
    res_source_only,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    scaler_m=scaler_m,
    scaler_s=scaler_s,
    seed=cfg.seed,
)

RECOMPUTE_SPLITS = False

FT_FRAC = 0.10
save_path = BASE_DIR_FD / "glacier_splits_FD.json"

#gl_level_split = ['IT', 'AT']
gl_level_split = []
id_level_split = ['IT', 'AT', 'SJM', 'ALA', 'CA_3', 'CA_4']

# ── Load existing glacier splits from disk ────────────────────────────────────
splits = {}
if not RECOMPUTE_SPLITS and save_path.exists():
    with open(save_path, "r") as f:
        splits = json.load(f)
    print(f"Loaded splits from: {save_path}")

# ── Only compute glacier splits for regions that need them ────────────────────
regions_needing_gl_split = [r for r in gl_level_split]
regions_to_compute = (regions_needing_gl_split if RECOMPUTE_SPLITS else
                      [r for r in regions_needing_gl_split if r not in splits])

for region in regions_to_compute:
    print(f"\n{'='*50}\nSplitting region: {region}")
    split = split_pool_holdout_sinkhorn(
        df_region=monthly_cache[region]["data_monthly"],
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
        scaler_m=scaler_m,
        scaler_s=scaler_s,
        blur_joint=blur_joint,
        pool_frac=0.2,
        n_restarts=50,
        seed=cfg.seed,
    )
    splits[region] = split
    print(f"  Pool glaciers    : {split['pool_glaciers']}")
    print(f"  Holdout glaciers : {split['holdout_glaciers']}")
    print(f"  Pool fraction    : {split['actual_pool_frac']:.1%}")
    print(f"  Sk(pool↔holdout) : {split['sinkhorn_pool_vs_holdout']:.4f}")

if regions_to_compute:
    with open(save_path, "w") as f:
        json.dump(splits, f, indent=2)
    print(f"\nSaved splits for: {regions_to_compute}")

In [ ]:
FORCE_RECOMPUTE = False
FORCE_RECOMPUTE_REGIONS = []

N_REPEATS = 5

# ── Build FT assets for all target subregions, repeated N_REPEATS times ──────
ft_assets_glacier = {}
ft_assets_id = {region: [] for region in TARGET_REGIONS_SUB}

for region in TARGET_REGIONS_SUB:
    print(f"\n{'='*50}\nBuilding FT assets for: {region}")

    df_monthly = monthly_cache[region]["data_monthly"]
    df_monthly_aug = monthly_cache[region]["data_monthly_aug"]
    head_pad = monthly_cache[region]["months_head_pad"]
    tail_pad = monthly_cache[region]["months_tail_pad"]
    force = FORCE_RECOMPUTE or (region in FORCE_RECOMPUTE_REGIONS)

    # 1. Glacier-level split (unchanged — deterministic by glacier, no repeats needed)
    if region in gl_level_split:
        if region in gl_level_split:
            pool_glaciers = splits[region]["pool_glaciers"]
            holdout_glaciers = splits[region]["holdout_glaciers"]
            exp_key = f"FD_to_{region}"

            ds_ft_gl = build_or_load_lstm_dataset_only(
                cfg=cfg,
                key=f"{exp_key}_FT",
                df_loss=df_monthly[df_monthly["GLACIER"].isin(pool_glaciers)],
                df_full=df_monthly_aug[df_monthly_aug["GLACIER"].isin(
                    pool_glaciers)],
                months_head_pad=head_pad,
                months_tail_pad=tail_pad,
                MONTHLY_COLS=MONTHLY_COLS,
                STATIC_COLS=STATIC_COLS,
                cache_dir=str(CACHE_DIR),
                force_recompute=force,
                kind="ft",
                show_progress=False,
            )
            ds_test_gl = build_or_load_lstm_dataset_only(
                cfg=cfg,
                key=f"{exp_key}_TEST",
                df_loss=df_monthly[df_monthly["GLACIER"].isin(
                    holdout_glaciers)],
                df_full=df_monthly_aug[df_monthly_aug["GLACIER"].isin(
                    holdout_glaciers)],
                months_head_pad=head_pad,
                months_tail_pad=tail_pad,
                MONTHLY_COLS=MONTHLY_COLS,
                STATIC_COLS=STATIC_COLS,
                cache_dir=str(CACHE_DIR),
                force_recompute=force,
                kind="test",
                show_progress=False,
            )
            ft_train_idx_gl, ft_val_idx_gl = mbm.data_processing.MBSequenceDataset.split_indices(
                len(ds_ft_gl), val_ratio=0.2, seed=cfg.seed)
            ft_assets_glacier[region] = {
                "ds_ft": ds_ft_gl,
                "ds_test": ds_test_gl,
                "ft_train_idx": ft_train_idx_gl,
                "ft_val_idx": ft_val_idx_gl,
            }
            print(
                f"  [glacier-split] ds_ft  : {len(ds_ft_gl)} seqs | ds_test: {len(ds_test_gl)} seqs"
            )

    # 2. ID-level split, repeated N_REPEATS times with different seeds
    if region in id_level_split:
        winter_ids_full = df_monthly[df_monthly["PERIOD"] ==
                                     "winter"]["ID"].unique().tolist()
        annual_ids_full = df_monthly[df_monthly["PERIOD"] ==
                                     "annual"]["ID"].unique().tolist()

        for rep in range(N_REPEATS):
            winter_ids = list(winter_ids_full)
            annual_ids = list(annual_ids_full)

            rep_seed = cfg.seed + rep
            rng = np.random.default_rng(rep_seed)
            rng.shuffle(winter_ids)
            rng.shuffle(annual_ids)

            n_ft_w = max(1, int(len(winter_ids) *
                                FT_FRAC)) if winter_ids else 0
            n_ft_a = max(1, int(len(annual_ids) * FT_FRAC))
            ft_ids = winter_ids[:n_ft_w] + annual_ids[:n_ft_a]
            test_ids = winter_ids[n_ft_w:] + annual_ids[n_ft_a:]

            # keep track of ft_ids per rep for the cross-rep check below
            if rep == 0:
                _ft_id_sets_for_check = []
            _ft_id_sets_for_check.append(set(ft_ids))

            exp_key_id = f"FD_to_{region}_IDsplit_rep{rep}"
            ds_ft_id = build_or_load_lstm_dataset_only(
                cfg=cfg,
                key=f"{exp_key_id}_FT",
                df_loss=df_monthly[df_monthly["ID"].isin(ft_ids)],
                df_full=df_monthly_aug[df_monthly_aug["ID"].isin(ft_ids)],
                months_head_pad=head_pad,
                months_tail_pad=tail_pad,
                MONTHLY_COLS=MONTHLY_COLS,
                STATIC_COLS=STATIC_COLS,
                cache_dir=str(CACHE_DIR),
                force_recompute=force,
                kind="ft",
                show_progress=False,
            )
            ds_test_id = build_or_load_lstm_dataset_only(
                cfg=cfg,
                key=f"{exp_key_id}_TEST",
                df_loss=df_monthly[df_monthly["ID"].isin(test_ids)],
                df_full=df_monthly_aug[df_monthly_aug["ID"].isin(test_ids)],
                months_head_pad=head_pad,
                months_tail_pad=tail_pad,
                MONTHLY_COLS=MONTHLY_COLS,
                STATIC_COLS=STATIC_COLS,
                cache_dir=str(CACHE_DIR),
                force_recompute=force,
                kind="test",
                show_progress=False,
            )
            ft_train_idx_id, ft_val_idx_id = mbm.data_processing.MBSequenceDataset.split_indices(
                len(ds_ft_id), val_ratio=0.2, seed=rep_seed)

            ft_assets_id[region].append({
                "rep": rep,
                "seed": rep_seed,
                "ds_ft": ds_ft_id,
                "ds_test": ds_test_id,
                "ft_train_idx": ft_train_idx_id,
                "ft_val_idx": ft_val_idx_id,
            })
            print(f"  [ID-split rep {rep}] ds_ft  : {len(ds_ft_id)} seqs | "
                  f"ds_test: {len(ds_test_id)} seqs")

            # ── sanity check after all reps for this region are built ────────────
        if region in id_level_split:
            n_reps = len(_ft_id_sets_for_check)
            identical_pairs = [
                (i, j) for i in range(n_reps) for j in range(i + 1, n_reps)
                if _ft_id_sets_for_check[i] == _ft_id_sets_for_check[j]
            ]
            if identical_pairs:
                print(
                    f"  WARNING [{region}]: identical FT id-sets for repeats {identical_pairs}"
                )
            else:
                overlaps = [
                    len(_ft_id_sets_for_check[i] & _ft_id_sets_for_check[j]) /
                    len(_ft_id_sets_for_check[i]) for i in range(n_reps)
                    for j in range(i + 1, n_reps)
                ]
                print(
                    f"  [check] {region}: {n_reps} repeats distinct (mean overlap {np.mean(overlaps):.1%})"
                )

# convenience dict: glacier-split where available, ID-split (list of reps) for the rest
ft_assets = {
    **ft_assets_glacier,
    **{
        r: ft_assets_id[r]
        for r in ft_assets_id if r not in ft_assets_glacier
    },
}

In [ ]:
models_dir_ft = BASE_DIR_FD / "models" / "finetuned"
os.makedirs(models_dir_ft, exist_ok=True)

ft_models_gl = {}  # glacier-split
ft_models_id = {}  # ID-split, now nested by rep

# ── glacier-split: unchanged structure ────────────────────────────────────────
for region in gl_level_split:
    ft_models_gl[region] = {}
    print(f"\n{'='*60}")
    print(f"Fine-tuning on: {region} (GLsplit)")

    for strategy in ["head_only", "full"]:
        ft_models_gl[region][strategy] = finetune_transformer_on_target(
            cfg=cfg,
            model_pooled=model_pooled,
            ds_ft=ft_assets_glacier[region]["ds_ft"],
            ds_pooled_scaler=ds_pooled_scaler,
            device=device,
            best_params=PARAMS_TRANSFORMER,
            model_filename=str(
                models_dir_ft /
                f"transformer_ft_{strategy}_{region}_{MODEL_DATE}.pt"),
            strategy=strategy,
            epochs=PARAMS_FT['epochs'],
            lr_factor=PARAMS_FT['lr_factor'],
            force_retrain=False,
        )
        print(f"  [{region}] {strategy} done")

# ── ID-split: loop over reps, one model set per rep ──────────────────────────
for region in id_level_split:
    ft_models_id[region] = []  # list of per-rep strategy dicts
    print(f"\n{'='*60}")
    print(f"Fine-tuning on: {region} (IDsplit, {N_REPEATS} repeats)")

    for rep_data in ft_assets_id[region]:
        rep = rep_data["rep"]
        rep_models = {}

        for strategy in ["head_only", "full", "adapter"]:
            rep_models[strategy] = finetune_transformer_on_target(
                cfg=cfg,
                model_pooled=model_pooled,
                ds_ft=rep_data["ds_ft"],
                ds_pooled_scaler=ds_pooled_scaler,
                device=device,
                best_params=PARAMS_TRANSFORMER,
                model_filename=str(
                    models_dir_ft /
                    f"transformer_ft_{strategy}_{region}_IDsplit_rep{rep}_{MODEL_DATE}.pt"
                ),
                strategy=strategy,
                epochs=PARAMS_FT['epochs'],
                lr_factor=PARAMS_FT['lr_factor'],
                force_retrain=False,
            )
            print(f"  [{region}] rep {rep} - {strategy} done")

        ft_models_id[region].append({"rep": rep, "models": rep_models})

### Eval metrics:

In [ ]:
def _add_nrmse(metrics, preds_df):
    """Add NRMSE (normalized by std of observed PMB) for annual and winter."""
    preds_clean = preds_df.dropna(subset=["target", "pred"])

    std_annual = preds_clean.loc[preds_clean["PERIOD"] == "annual",
                                 "target"].std()
    std_winter = preds_clean.loc[preds_clean["PERIOD"] == "winter",
                                 "target"].std()

    metrics["NRMSE_annual"] = (metrics["RMSE_annual"] / std_annual
                               if std_annual and std_annual > 0 else np.nan)
    metrics["NRMSE_winter"] = (metrics["RMSE_winter"] / std_winter
                               if std_winter and std_winter > 0 else np.nan)
    return metrics

In [ ]:
eval_configs = []
for region in TARGET_REGIONS_SUB:
    ds_test_full = build_or_load_lstm_dataset_only(
        cfg=cfg,
        key=f"ZEROSHOT_{region}_FULL",
        df_loss=monthly_cache[region]["data_monthly"],
        df_full=monthly_cache[region]["data_monthly_aug"],
        months_head_pad=monthly_cache[region]["months_head_pad"],
        months_tail_pad=monthly_cache[region]["months_tail_pad"],
        MONTHLY_COLS=MONTHLY_COLS,
        STATIC_COLS=STATIC_COLS,
        cache_dir=str(CACHE_DIR),
        force_recompute=False,
        kind="test",
        show_progress=False,
    )
    eval_configs.append((region, {"ds_test": ds_test_full}))


def _eval_on(model, ds_test, scaler):
    ds_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        ds_test)
    test_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
        ds_copy, scaler, batch_size=128, seed=cfg.seed)
    return model.evaluate_with_preds(device, test_dl, ds_copy)


df_metrics_comparison = {}
region_pred_target_ranges = {
}  # {region: {model_label: {"y_min":, "y_max":, "pred_min":, "pred_max":}}}

for tgt_label, assets in eval_configs:
    if assets is None:
        print(f"No assets for {tgt_label}, skipping.")
        continue

    df_metrics_comparison[tgt_label] = {}
    region_pred_target_ranges[tgt_label] = {}

    m, preds_df = _eval_on(model_pooled, assets["ds_test"], ds_pooled_scaler)
    m = _add_nrmse(m, preds_df)
    df_metrics_comparison[tgt_label]["pooled_zs"] = m
    preds_clean = preds_df.dropna(subset=["target", "pred"])
    region_pred_target_ranges[tgt_label]["pooled_zs"] = {
        "y_min": float(preds_clean["target"].min()),
        "y_max": float(preds_clean["target"].max()),
        "pred_min": float(preds_clean["pred"].min()),
        "pred_max": float(preds_clean["pred"].max()),
    }

    for src, model in single_source_models.items():
        m, preds_df = _eval_on(model, assets["ds_test"],
                               single_source_scalers[src])
        m = _add_nrmse(m, preds_df)
        df_metrics_comparison[tgt_label][src] = m
        preds_clean = preds_df.dropna(subset=["target", "pred"])
        region_pred_target_ranges[tgt_label][src] = {
            "y_min": float(preds_clean["target"].min()),
            "y_max": float(preds_clean["target"].max()),
            "pred_min": float(preds_clean["pred"].min()),
            "pred_max": float(preds_clean["pred"].max()),
        }

# --- print summary ---
print(f"\n{'='*90}")
print(
    f"{'Model':<15} {'RMSE_a':>8} {'NRMSE_a':>8} {'RMSE_w':>8} {'NRMSE_w':>8} {'R2_a':>8} {'R2_w':>8}"
)
print(f"{'='*90}")
for tgt_label, results in df_metrics_comparison.items():
    print(f"\n--- {tgt_label} ---")
    for model_label, m in results.items():
        rng = region_pred_target_ranges[tgt_label][model_label]
        print(f"  {model_label:<15}  "
              f"RMSE_a={m['RMSE_annual']:.3f}  "
              f"NRMSE_a={m['NRMSE_annual']:.3f}  "
              f"RMSE_w={m['RMSE_winter']:.3f}  "
              f"NRMSE_w={m['NRMSE_winter']:.3f}  "
              f"R2_a={m['R2_annual']:.3f}  "
              f"R2_w={m['R2_winter']:.3f}  "
              f"y∈[{rng['y_min']:.2f}, {rng['y_max']:.2f}]  "
              f"pred∈[{rng['pred_min']:.2f}, {rng['pred_max']:.2f}]")

In [ ]:
# ── Fine-tuning metrics: evaluate no_ft + each FT strategy on the SAME per-rep test set ──
import numpy as np

ft_strategies = ["head_only", "full"]

for region in id_level_split:
    if region not in ft_assets_id or not ft_assets_id[region]:
        continue

    # find the model entry for each rep
    rep_model_lookup = {
        entry["rep"]: entry["models"]
        for entry in ft_models_id[region]
    }

    per_strategy_metrics = {label: [] for label in ["no_ft"] + ft_strategies}
    per_strategy_ranges = {label: [] for label in ["no_ft"] + ft_strategies}

    for rep_data in ft_assets_id[region]:
        rep = rep_data["rep"]
        ds_test_rep = rep_data["ds_test"]

        # zero-shot pooled model, evaluated on this rep's matched test set
        m, preds_df = _eval_on(model_pooled, ds_test_rep, ds_pooled_scaler)
        m = _add_nrmse(m, preds_df)
        per_strategy_metrics["no_ft"].append(m)
        preds_clean = preds_df.dropna(subset=["target", "pred"])
        per_strategy_ranges["no_ft"].append({
            "y_min":
            float(preds_clean["target"].min()),
            "y_max":
            float(preds_clean["target"].max()),
            "pred_min":
            float(preds_clean["pred"].min()),
            "pred_max":
            float(preds_clean["pred"].max()),
        })

        # each FT strategy, evaluated on the same rep's test set
        rep_models = rep_model_lookup.get(rep, {})
        for strategy in ft_strategies:
            if strategy not in rep_models:
                continue
            m, preds_df = _eval_on(rep_models[strategy], ds_test_rep,
                                   ds_pooled_scaler)
            m = _add_nrmse(m, preds_df)
            per_strategy_metrics[strategy].append(m)
            preds_clean = preds_df.dropna(subset=["target", "pred"])
            per_strategy_ranges[strategy].append({
                "y_min":
                float(preds_clean["target"].min()),
                "y_max":
                float(preds_clean["target"].max()),
                "pred_min":
                float(preds_clean["pred"].min()),
                "pred_max":
                float(preds_clean["pred"].max()),
            })

    # aggregate across reps: mean + std for each metric, store under the same
    # df_metrics_comparison[region] dict used by the zero-shot table/plot
    metric_keys = [
        "RMSE_annual", "NRMSE_annual", "RMSE_winter", "NRMSE_winter",
        "R2_annual", "R2_winter"
    ]

    for label, metrics_list in per_strategy_metrics.items():
        if not metrics_list:
            continue
        agg = {}
        for k in metric_keys:
            vals = [
                m[k] for m in metrics_list if k in m and not np.isnan(m[k])
            ]
            agg[k] = float(np.mean(vals)) if vals else np.nan
            agg[f"{k}_std"] = float(np.std(vals)) if len(vals) > 1 else 0.0
        df_metrics_comparison[region][label] = agg

        ranges_list = per_strategy_ranges[label]
        region_pred_target_ranges[region][label] = {
            "y_min": min(r["y_min"] for r in ranges_list),
            "y_max": max(r["y_max"] for r in ranges_list),
            "pred_min": min(r["pred_min"] for r in ranges_list),
            "pred_max": max(r["pred_max"] for r in ranges_list),
        }

    print(f"  [{region}] aggregated over {len(ft_assets_id[region])} reps: "
          f"{list(per_strategy_metrics.keys())}")

# --- extend print summary to include FT strategies ---
print(f"\n{'='*90}")
print(
    f"{'Model':<15} {'RMSE_a':>8} {'NRMSE_a':>8} {'RMSE_w':>8} {'NRMSE_w':>8} {'R2_a':>8} {'R2_w':>8}"
)
print(f"{'='*90}")
for tgt_label, results in df_metrics_comparison.items():
    print(f"\n--- {tgt_label} ---")
    for model_label, m in results.items():
        rng = region_pred_target_ranges[tgt_label][model_label]
        print(f"  {model_label:<15}  "
              f"RMSE_a={m['RMSE_annual']:.3f}  "
              f"NRMSE_a={m['NRMSE_annual']:.3f}  "
              f"RMSE_w={m['RMSE_winter']:.3f}  "
              f"NRMSE_w={m['NRMSE_winter']:.3f}  "
              f"R2_a={m['R2_annual']:.3f}  "
              f"R2_w={m['R2_winter']:.3f}  "
              f"y∈[{rng['y_min']:.2f}, {rng['y_max']:.2f}]  "
              f"pred∈[{rng['pred_min']:.2f}, {rng['pred_max']:.2f}]")

## Distance - comparison of models:

### Pooled:

In [ ]:
res_source_only = {
    region: {
        "df_train": res_xreg_by_source[region]["df_train"],
    }
    for region in SOURCE_REGIONS
}


def add_cartesian_coords(df, lon_col="POINT_LON", lat_col="POINT_LAT"):
    """Add unit-sphere Cartesian coordinates to avoid longitude wraparound issues."""
    df = df.copy()
    lat_rad = np.radians(df[lat_col])
    lon_rad = np.radians(df[lon_col])
    df["COORD_X"] = np.cos(lat_rad) * np.cos(lon_rad)
    df["COORD_Y"] = np.cos(lat_rad) * np.sin(lon_rad)
    df["COORD_Z"] = np.sin(lat_rad)
    return df


COORD_COLS = ['COORD_X', 'COORD_Y', 'COORD_Z']
# NOTE: coords are now a standalone "geo" channel, not folded into static/topo
STATIC_COLS_WITH_COORDS = STATIC_COLS  # unchanged — kept the name for compatibility downstream

# apply Cartesian projection to source-only training/test data used for scalers
res_source_only_coords = {
    k: {
        **v,
        "df_train":
        add_cartesian_coords(v["df_train"]),
        **({
            "df_test": add_cartesian_coords(v["df_test"])
        } if "df_test" in v else {}),
    }
    for k, v in res_source_only.items()
}

scaler_m, scaler_s, scaler_joint, scaler_geo = build_global_scalers_multi_source_simple(
    res_source_only_coords,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS_WITH_COORDS,
    geo_cols=COORD_COLS,
)
blur_m, blur_s, blur_joint, blur_geo = estimate_global_bandwidths_simple(
    res_source_only_coords,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS_WITH_COORDS,
    geo_cols=COORD_COLS,
    scaler_m=scaler_m,
    scaler_s=scaler_s,
    scaler_geo=scaler_geo,
    seed=cfg.seed,
)

RECOMPUTE_SHIFTS_POOLED = False
SHIFTS_CACHE_POOLED = BASE_DIR_FD / "domain_shifts_pooled_cache.pkl"

bandwidths_m = [blur_m * 0.5, blur_m, blur_m * 2.0]
bandwidths_s = [blur_s * 0.5, blur_s, blur_s * 2.0]
bandwidths_geo = [blur_geo * 0.5, blur_geo, blur_geo * 2.0]
scaler_joint = scaler_joint  # already computed in scalers cell

#EXCLUDE_REGIONS = {'CENTRALASIA', 'ALA'}
EXCLUDE_REGIONS = {'CENTRALASIA', 'ALA_2', 'ALA_4', 'ALA_6'}
tgt_labels = [
    t for t in df_metrics_comparison.keys() if t not in EXCLUDE_REGIONS
]

if RECOMPUTE_SHIFTS_POOLED and SHIFTS_CACHE_POOLED.exists():
    SHIFTS_CACHE_POOLED.unlink()
    print(f"Deleted existing cache: {SHIFTS_CACHE_POOLED}")

if SHIFTS_CACHE_POOLED.exists():
    with open(SHIFTS_CACHE_POOLED, "rb") as f:
        cache = pickle.load(f)
    all_shifts_pooled = cache["all_shifts_pooled"]
    print(f"Loaded pooled shifts from cache: {SHIFTS_CACHE_POOLED}")

else:
    # df_src = full pooled training data (all source regions combined)
    df_src_pooled = pd.concat(
        [res_xreg_by_source[r]["df_train"] for r in SOURCE_REGIONS],
        ignore_index=True,
    )
    df_src_pooled = add_cartesian_coords(df_src_pooled)

    all_shifts_pooled = {}
    for region in tqdm(tgt_labels, desc="Pooled domain shifts"):
        df_tgt = add_cartesian_coords(monthly_cache[region]["data_monthly"])
        shift = compute_domain_shift(
            df_src=df_src_pooled,
            df_tgt=df_tgt,
            monthly_cols=MONTHLY_COLS,
            static_cols=STATIC_COLS_WITH_COORDS,
            geo_cols=COORD_COLS,
            scaler_m=scaler_m,
            scaler_s=scaler_s,
            scaler_geo=scaler_geo,
            scaler_joint=scaler_joint,
            blur_m=blur_m,
            blur_s=blur_s,
            blur_geo=blur_geo,
            blur_joint=blur_joint,
            bandwidths_m=bandwidths_m,
            bandwidths_s=bandwidths_s,
            bandwidths_geo=bandwidths_geo,
            seed=cfg.seed,
        )
        all_shifts_pooled[region] = shift
        print(f"  {region}: joint={shift['D_sinkhorn_joint']:.4f} "
              f"clim={shift['D_sinkhorn_climate']:.4f} "
              f"topo={shift['D_sinkhorn_topo']:.4f} "
              f"geo={shift['D_sinkhorn_geo']:.4f}")

    with open(SHIFTS_CACHE_POOLED, "wb") as f:
        pickle.dump({"all_shifts_pooled": all_shifts_pooled}, f)
    print(f"Saved pooled shifts to cache: {SHIFTS_CACHE_POOLED}")

### Single regions:

In [ ]:
RECOMPUTE_SHIFTS = False

SHIFTS_CACHE = BASE_DIR_XREG / "domain_shifts_cache_cross_region.pkl"

res_all_xreg_by_source = {}
ALL_TARGETS = SOURCE_REGIONS + TARGET_REGIONS_SUB

for src_region in SOURCE_REGIONS:
    target_codes = [t for t in ALL_TARGETS if t not in src_region]
    res_all_xreg = build_xreg_res_all(
        res_xreg=res_xreg_by_source[src_region],
        target_source_codes=target_codes,
        source_col="SOURCE_CODE",
        key_prefix=f"XREG_{src_region}_TO",
        ch_code=src_region,
        region_groups={},
    )
    res_all_xreg_by_source[src_region] = res_all_xreg

if RECOMPUTE_SHIFTS and SHIFTS_CACHE.exists():
    SHIFTS_CACHE.unlink()
    print(f"Deleted existing cache: {SHIFTS_CACHE}")

if SHIFTS_CACHE.exists():
    with open(SHIFTS_CACHE, "rb") as f:
        cache = pickle.load(f)
    scaler_m = cache["scaler_m"]
    scaler_s = cache["scaler_s"]
    scaler_geo = cache["scaler_geo"]
    blur_m = cache["blur_m"]
    blur_s = cache["blur_s"]
    blur_geo = cache["blur_geo"]
    blur_joint = cache["blur_joint"]
    bandwidths_m = cache["bandwidths_m"]
    bandwidths_s = cache["bandwidths_s"]
    bandwidths_geo = cache["bandwidths_geo"]
    all_shifts_by_source = cache["all_shifts_by_source"]
    print(f"Blurs from cache: blur_m={blur_m:.4f}, blur_s={blur_s:.4f}, "
          f"blur_geo={blur_geo:.4f}, blur_joint={blur_joint:.4f}")
    print(f"Loaded shifts from cache: {SHIFTS_CACHE}")

else:
    COORD_COLS = ['COORD_X', 'COORD_Y', 'COORD_Z']
    STATIC_COLS_WITH_COORDS = STATIC_COLS  # geo is now its own channel, not folded into static/topo

    # apply Cartesian projection to source data used for fitting scalers
    res_xreg_by_source_coords = {
        k: {
            **v,
            "df_train":
            add_cartesian_coords(v["df_train"]),
            **({
                "df_test": add_cartesian_coords(v["df_test"])
            } if "df_test" in v else {}),
        }
        for k, v in res_xreg_by_source.items()
    }

    scaler_m, scaler_s, scaler_joint, scaler_geo = build_global_scalers_multi_source_simple(
        res_xreg_by_source_coords,
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS_WITH_COORDS,
        geo_cols=COORD_COLS,
    )

    blur_m, blur_s, blur_joint, blur_geo = estimate_global_bandwidths_simple(
        res_xreg_by_source_coords,
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS_WITH_COORDS,
        geo_cols=COORD_COLS,
        scaler_m=scaler_m,
        scaler_s=scaler_s,
        scaler_geo=scaler_geo,
        seed=cfg.seed,
    )
    bandwidths_m = [blur_m * 0.5, blur_m, blur_m * 2.0]
    bandwidths_s = [blur_s * 0.5, blur_s, blur_s * 2.0]
    bandwidths_geo = [blur_geo * 0.5, blur_geo, blur_geo * 2.0]
    print(f"Estimated blurs: blur_m={blur_m:.4f}, blur_s={blur_s:.4f}, "
          f"blur_geo={blur_geo:.4f}, blur_joint={blur_joint:.4f}")

    all_shifts_by_source = {}
    for src_region in SOURCE_REGIONS:
        all_shifts = {}
        for key in tqdm(res_all_xreg_by_source[src_region], desc=src_region):
            df_src = add_cartesian_coords(
                res_all_xreg_by_source[src_region][key]["df_train"])
            df_tgt = add_cartesian_coords(
                res_all_xreg_by_source[src_region][key]["df_test"])
            shift = compute_domain_shift(
                df_src=df_src,
                df_tgt=df_tgt,
                monthly_cols=MONTHLY_COLS,
                static_cols=STATIC_COLS_WITH_COORDS,
                geo_cols=COORD_COLS,
                scaler_m=scaler_m,
                scaler_s=scaler_s,
                scaler_geo=scaler_geo,
                scaler_joint=scaler_joint,
                blur_m=blur_m,
                blur_s=blur_s,
                blur_geo=blur_geo,
                blur_joint=blur_joint,
                bandwidths_m=bandwidths_m,
                bandwidths_s=bandwidths_s,
                bandwidths_geo=bandwidths_geo,
                seed=cfg.seed,
            )
            all_shifts[key] = shift
        all_shifts_by_source[src_region] = all_shifts

    with open(SHIFTS_CACHE, "wb") as f:
        pickle.dump(
            {
                "scaler_m": scaler_m,
                "scaler_s": scaler_s,
                "scaler_geo": scaler_geo,
                "blur_m": blur_m,
                "blur_s": blur_s,
                "blur_geo": blur_geo,
                "blur_joint": blur_joint,
                "bandwidths_m": bandwidths_m,
                "bandwidths_s": bandwidths_s,
                "bandwidths_geo": bandwidths_geo,
                "all_shifts_by_source": all_shifts_by_source,
            }, f)
    print(f"Saved shifts to cache: {SHIFTS_CACHE}")

### Plots:

In [ ]:
TF_COLOR = NATURE_PALETTE["sky_blue"]


def plot_pooled_performance_vs_shift(
        all_shifts_pooled: dict,
        df_metrics: dict,
        metric: str = "RMSE_annual",
        model_label: str = "no_ft",  # zero-shot entry in df_ft_metrics_id
        distance_metrics: list[str] = [
            "D_sinkhorn_climate", "D_sinkhorn_topo", "D_sinkhorn_geo",
            "D_sinkhorn_joint"
        ],
        model: str = "Transformer",
        src='Pooled'):
    # ── 1. Build merged DataFrame ─────────────────────────────────────────────
    shift_rows = []
    for tgt, shift in all_shifts_pooled.items():
        row = {"target": tgt}
        row.update(shift)
        shift_rows.append(row)
    df_shifts = pd.DataFrame(shift_rows)

    perf_rows = []
    for tgt, results in df_metrics.items():
        if model_label not in results:
            continue
        row = {"target": tgt, metric: results[model_label][metric]}
        std_key = f"{metric}_std"
        if std_key in results[model_label]:
            row[std_key] = results[model_label][std_key]
        perf_rows.append(row)
    df_perf = pd.DataFrame(perf_rows)

    df_plot = df_shifts.merge(df_perf, on="target")

    # ── 2. Plot ───────────────────────────────────────────────────────────────
    dist_labels = {
        "D_sinkhorn_climate": "Sinkhorn distance (climate)",
        "D_sinkhorn_topo": "Sinkhorn distance (topographic)",
        "D_sinkhorn_geo": "Sinkhorn distance (geographic)",
        "D_sinkhorn_joint": "Sinkhorn distance (joint)",
        "D_mmd2_climate": "MMD² (climate)",
        "D_mmd2_topo": "MMD² (topographic)",
        "D_mmd2_geo": "MMD² (geographic)",
        "D_energy_climate": "Energy distance (climate)",
        "D_energy_topo": "Energy distance (topographic)",
        "D_energy_geo": "Energy distance (geographic)",
    }

    n = len(distance_metrics)
    fig, axes = plt.subplots(
        1,
        n,
        figsize=(4.5 * n, 4),
        constrained_layout=False,  # turn off auto-redistribution
    )
    fig.subplots_adjust(left=0.08,
                        right=0.98,
                        bottom=0.18,
                        top=0.85,
                        wspace=0.3)
    if n == 1:
        axes = [axes]

    std_key = f"{metric}_std"
    has_err = std_key in df_plot.columns

    for ax, dist_key in zip(axes, distance_metrics):
        x = df_plot[dist_key].values
        y = df_plot[metric].values
        labels = df_plot["target"].values
        yerr = df_plot[std_key].values if has_err else None

        ax.errorbar(
            x,
            y,
            yerr=yerr,
            fmt="o",
            color=TF_COLOR,
            markersize=8,
            zorder=3,
            ecolor="0.5",
            elinewidth=1.0,
            capsize=3,
            markeredgecolor="white",
            markeredgewidth=0.5,
        )

        for xi, yi, lab in zip(x, y, labels):
            ax.annotate(
                f"{src} \u2192 {lab}",
                (xi, yi),
                xytext=(5, 4),
                textcoords="offset points",
                fontsize=8,
                color="0.3",
            )

        # leave headroom above the highest point for its annotation
        y_min, y_max = y.min(), y.max()
        y_range = y_max - y_min if y_max > y_min else 1.0
        ax.set_ylim(y_min - y_range * 0.1, y_max + y_range * 0.5)

        ax.set_xlabel(dist_labels.get(dist_key, dist_key), fontsize=9)
        ax.set_ylabel(
            f"{model} {metric} (m w.e.)"
            if dist_key == distance_metrics[0] else "",
            fontsize=9,
        )
        ax.set_title(dist_key)

    return fig

#### Annual RMSE:

In [ ]:
fig = plot_pooled_performance_vs_shift(
    all_shifts_pooled,
    df_metrics_comparison,
    metric="RMSE_annual",
    model_label=
    "pooled_zs",  # was "no_ft" — that key doesn't exist in df_metrics_comparison
    distance_metrics=[
        "D_sinkhorn_climate", "D_sinkhorn_topo", "D_sinkhorn_geo"
    ])
plt.show()

for src in SOURCE_REGIONS:
    fig = plot_pooled_performance_vs_shift(
        all_shifts_pooled,
        df_metrics_comparison,
        metric="RMSE_annual",
        model_label=
        src,  # was "no_ft" — that key doesn't exist in df_metrics_comparison
        src=src,
        distance_metrics=[
            "D_sinkhorn_climate", "D_sinkhorn_topo", "D_sinkhorn_geo"
        ])
    fig.savefig(f'figures/paperTF/appendix/app_ds_{src}.png')
    plt.show()

#### Annual gain:

In [ ]:
ft_strategy_for_gain = "full"
gain_label = f"ft_gain_{ft_strategy_for_gain}"

gain_metric_keys = [
    "RMSE_annual", "NRMSE_annual", "RMSE_winter", "NRMSE_winter"
]

for region in id_level_split:
    if region not in df_metrics_comparison:
        continue
    results = df_metrics_comparison[region]
    if "no_ft" not in results or ft_strategy_for_gain not in results:
        continue

    no_ft = results["no_ft"]
    ft = results[ft_strategy_for_gain]

    gain = {}
    for k in gain_metric_keys:
        # positive gain = error reduction from fine-tuning
        gain[f"{k}_gain"] = no_ft[k] - ft[k]
        # approximate uncertainty propagation (NOT a paired diff — see caveat below)
        std_no_ft = no_ft.get(f"{k}_std", 0.0)
        std_ft = ft.get(f"{k}_std", 0.0)
        gain[f"{k}_gain_std"] = float(np.sqrt(std_no_ft**2 + std_ft**2))

    df_metrics_comparison[region][gain_label] = gain

In [ ]:
fig = plot_pooled_performance_vs_shift(all_shifts_pooled,
                                       df_metrics_comparison,
                                       metric="RMSE_annual_gain",
                                       model_label="ft_gain_full",
                                       distance_metrics=[
                                           "D_sinkhorn_climate",
                                           "D_sinkhorn_topo", "D_sinkhorn_geo"
                                       ])

# save
fig.savefig('figures/paperTF/appendix/app_ds_pooled.png')
plt.show()